In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL X-CAL — EXPERIMENT 1: CALIBRATION ANALYSIS
#   Reliability diagram + Brier + ECE + calibration slope, all four units,
#   plus the per-BI-RADS score-to-risk table that motivates Novelty 2.
#   No GPU. ~10 seconds.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

D    = r"/root/autodl-tmp/CBIS"
OUT  = os.path.join(D, "figs"); os.makedirs(OUT, exist_ok=True)
stem = lambda q: os.path.splitext(os.path.basename(str(q)))[0]

d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv")); d["_k"] = d["img"].map(stem)
m = pd.read_csv(os.path.join(D, "cv_mass_twostream_officialsplit_oof.csv"))
m["_k"] = m["img"].map(stem)
T = d.merge(m[["_k", "prob"]], on="_k")
T["y"] = T.label.astype(int); T["a"] = pd.to_numeric(T.assessment, errors="coerce").fillna(4).astype(int)
if "side" not in T.columns:
    fx = pd.read_csv(os.path.join(D, "cbis_mass_fixed.csv"))
    fx["_k"] = fx["cropped image file path"].map(stem)
    T = T.merge(fx[["_k", "side"]].drop_duplicates("_k"), on="_k", how="left")
T["breast_key"] = T.patient_id.astype(str) + "_" + T["side"].astype(str)

def unit(key):
    if key is None: return T.y.values.astype(float), T.prob.values, T.a.values
    g = T.groupby(key).agg(y=("y","max"), p=("prob","mean"), a=("a","max"))
    return g.y.values.astype(float), g.p.values, g.a.values

brier = lambda y,p: float(np.mean((p - y) ** 2))

def ece(y, p, B=10):
    e, n, edges = 0.0, len(y), np.linspace(0, 1, B + 1)
    for i in range(B):
        s = (p >= edges[i]) & (p <= edges[i+1]) if i == 0 else (p > edges[i]) & (p <= edges[i+1])
        if s.sum(): e += s.sum() / n * abs(y[s].mean() - p[s].mean())
    return float(e)

def slope_int(y, p):
    z = np.log(np.clip(p, 1e-6, 1-1e-6) / (1 - np.clip(p, 1e-6, 1-1e-6)))
    X, b = np.column_stack([np.ones_like(z), z]), np.array([0.0, 1.0])
    for _ in range(100):
        mu = 1 / (1 + np.exp(-X @ b)); W = mu * (1 - mu)
        try: step = np.linalg.solve(-(X * W[:, None]).T @ X, -(X.T @ (y - mu)))
        except np.linalg.LinAlgError: break
        b = b + step
        if np.max(np.abs(step)) < 1e-9: break
    return float(b[1]), float(b[0])

UNITS = [("ROI", None), ("Lesion", "lesion_key"), ("Breast", "breast_key"), ("Patient", "patient_id")]
print("=" * 74); print("EXPERIMENT 1 — CALIBRATION, official test partition"); print("=" * 74)
print("  %-8s %5s %8s %8s %8s %8s %10s" % ("unit","n","AUC","Brier","ECE","slope","intercept"))
for nm, key in UNITS:
    y, p, _ = unit(key); s, i0 = slope_int(y, p)
    print("  %-8s %5d %8.4f %8.4f %8.4f %8.3f %10.3f"
          % (nm, len(y), roc_auc_score(y, p), brier(y, p), ece(y, p), s, i0))
print("\n  perfect calibration: slope 1.000, intercept 0.000, ECE 0")

y, p, a = unit("lesion_key")
print("\n" + "=" * 74)
print("SCORE-TO-RISK MAPPING BY BI-RADS  (this is what motivates Novelty 2)")
print("=" * 74)
print("  %-8s %4s %11s %11s %8s" % ("BI-RADS","n","mean score","observed","ratio"))
for c in sorted(set(a.tolist())):
    s = a == c
    r = y[s].mean() / p[s].mean() if p[s].mean() > 0 else float("nan")
    print("  %-8d %4d %11.3f %11.3f %8.2f" % (c, s.sum(), p[s].mean(), y[s].mean(), r))

z  = np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))
sl, ic = slope_int(y, p)
pc = 1 / (1 + np.exp(-(ic + sl * z)))
grid = np.arange(0.01, 0.995, 0.005)
b4 = max(((p  >= t).astype(int) == y).mean() for t in grid)
af = max(((pc >= t).astype(int) == y).mean() for t in grid)
print("\n  global Platt recalibration is monotone, so it changes nothing a")
print("  single threshold can do:")
print("    AUC                          before %.4f   after %.4f" % (roc_auc_score(y,p), roc_auc_score(y,pc)))
print("    best single-threshold acc    before %.4f   after %.4f" % (b4, af))

# ── reliability diagram, lesion level, black and white ────────────────────
q = np.quantile(p, np.linspace(0, 1, 11)); q[0] -= 1e-9
xs, ys, ns = [], [], []
for i in range(10):
    s = (p > q[i]) & (p <= q[i+1])
    if s.sum(): xs.append(p[s].mean()); ys.append(y[s].mean()); ns.append(int(s.sum()))

fig, ax = plt.subplots(figsize=(4.4, 4.4))
ax.plot([0, 1], [0, 1], color="0.55", lw=1.0, ls="--", zorder=1)
ax.plot(xs, ys, color="black", lw=1.2, marker="o", ms=5, mfc="white", mec="black", zorder=3)
ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed malignancy rate")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
ax.set_xticks(np.arange(0, 1.01, 0.2)); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.text(0.04, 0.93, "Lesion level, n=%d\nBrier %.4f   ECE %.3f\nslope %.2f   intercept %.2f"
        % (len(y), brier(y,p), ece(y,p), sl, ic), fontsize=8, va="top", family="monospace")
ax.text(0.62, 0.10, "perfect\ncalibration", fontsize=7, color="0.45", style="italic")
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.tick_params(labelsize=8); ax.grid(True, lw=0.4, color="0.88", zorder=0)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "FigCal_reliability.png"), dpi=400)
fig.savefig(os.path.join(OUT, "FigCal_reliability.pdf"))
print("\n  saved -> %s/FigCal_reliability.{png,pdf}" % OUT)
print("  bin counts:", ns)